# Tests

> Behavioral tests for the Rugprint object API and projection-rug plotting logic.

These tests are designed for nbdev CI. They avoid pixel-perfect image comparison and instead check the stable structure of the object model, edge planning, highlighting, and matplotlib output.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from rugprint.core import (
    Rugprint,
    default_penguins_layout,
    load,
    load_penguins,
    rank_pair_separation,
    rugprint,
)
from rugprint.core import _normalize_rug_edges

## Test Fixtures

Use a tiny deterministic dataframe for most tests, and use Palmer Penguins only where the demo behavior itself matters.

In [ ]:
sample = pd.DataFrame(
    {
        "a": [1, 2, 3, 4, 5, 6],
        "b": [1, 2, 2, 5, 6, 7],
        "c": [7, 6, 5, 3, 2, 1],
        "species": ["x", "x", "x", "y", "y", "y"],
    },
    index=["p0", "p1", "p2", "p3", "p4", "p5"],
)

projections = [("a", "b"), ("a", "c"), ("b", "c")]
layout = {
    ("a", "b"): (0, 1),
    ("a", "c"): (1, 1),
    ("b", "c"): (1, 2),
}

## Demo Data And Layout

In [ ]:
penguins = load_penguins()
features = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]

assert set(features + ["species"]).issubset(penguins.columns)
assert penguins[features + ["species"]].isna().sum().sum() == 0
assert len(penguins) > 300

penguin_projections, penguin_layout = default_penguins_layout()
assert len(penguin_projections) == 4
assert set(penguin_projections) == set(penguin_layout)

# The demo is intentionally arranged as a chain of adjacent shared-variable projections.
for left, right in zip(penguin_projections, penguin_projections[1:]):
    assert set(left).intersection(right)
    r1, c1 = penguin_layout[left]
    r2, c2 = penguin_layout[right]
    assert abs(r1 - r2) + abs(c1 - c2) == 1

## Object Construction

In [ ]:
rp = load(
    sample,
    projections=projections,
    layout=layout,
    group="species",
    highlight="p0",
    title="Test Rugprint",
)

assert isinstance(rp, Rugprint)
assert rp.data is sample
assert rp.projections == projections
assert rp.layout == layout
assert rp.group == "species"
assert rp.highlight == "p0"
assert rp.plot_data.equals(sample)
assert rp.highlight_row.name == "p0"
assert rp.highlight_row["a"] == 1

rp2 = rp.with_highlight("p4")
assert isinstance(rp2, Rugprint)
assert rp2 is not rp
assert rp2.highlight == "p4"
assert rp.highlight == "p0"
assert rp2.projections == rp.projections
assert rp2.layout == rp.layout

## Ranking Projection Pairs

In [ ]:
ranked = rp.rank_pairs(["a", "b", "c"])
assert list(ranked.columns) == ["x", "y", "mean_centroid_distance"]
assert len(ranked) == 3
assert ranked["mean_centroid_distance"].is_monotonic_decreasing

ranked_direct = rank_pair_separation(sample, ["a", "b", "c"], group="species")
pd.testing.assert_frame_equal(ranked, ranked_direct)

## Rug Edge Planning

In [ ]:
minimal = _normalize_rug_edges("minimal", projections, layout)
assert all(edges == {"x": "bottom", "y": "left"} for edges in minimal.values())

shared = _normalize_rug_edges("shared", projections, layout)
assert shared[("a", "b")]["x"] == "bottom"
assert shared[("a", "c")]["x"] == "top"
assert shared[("a", "c")]["y"] == "right"
assert shared[("b", "c")]["y"] == "left"

outer = _normalize_rug_edges("outer", projections, layout)
assert outer[("a", "b")]["y"] == "left"
assert outer[("b", "c")]["y"] == "right"

custom = _normalize_rug_edges(
    {
        ("a", "b"): ("top", "right"),
        ("a", "c"): ("bottom",),
        ("b", "c"): "left",
    },
    projections,
    layout,
)
assert custom[("a", "b")] == {"x": "top", "y": "right"}
assert custom[("a", "c")] == {"x": "bottom", "y": None}
assert custom[("b", "c")] == {"x": None, "y": "left"}

## Plot Structure

In [ ]:
fig = rp.plot()
assert fig.__class__.__name__ == "Figure"
assert len(fig.axes) == len(projections)

# Diagram mode hides axis and tick labels by default.
for ax in fig.axes:
    assert ax.get_xlabel() == ""
    assert ax.get_ylabel() == ""
    assert not any(label.get_visible() for label in ax.get_xticklabels())
    assert not any(label.get_visible() for label in ax.get_yticklabels())

# Highlight connectors between adjacent shared-variable panels are figure-level artists.
assert len(fig.artists) >= 2
plt.close(fig)

## Plot Options

In [ ]:
fig = rp.plot(show_axis_labels=True, show_tick_labels=True, rug_edges="minimal", connect_shared_rugs=False)
assert len(fig.axes) == len(projections)
assert len(fig.artists) == 0
assert any(ax.texts for ax in fig.axes)
assert any(label.get_visible() for ax in fig.axes for label in ax.get_xticklabels())
plt.close(fig)

fig = rp.plot(highlight=None)
assert len(fig.artists) == 0
plt.close(fig)

fig = rugprint(sample, projections=[("a", "b")], layout={("a", "b"): (0, 0)}, group="species")
assert fig.__class__.__name__ == "Figure"
assert len(fig.axes) == 1
plt.close(fig)

## Error Paths

In [ ]:
try:
    load(sample, projections=[("a", "missing")])
except KeyError as err:
    assert "missing" in str(err)
else:
    raise AssertionError("Expected missing projection column to raise KeyError")

try:
    load(sample, projections=[("a", "b"), ("a", "c")], layout={("a", "b"): (0, 0)})
except KeyError as err:
    assert "Missing layout" in str(err)
else:
    raise AssertionError("Expected incomplete layout to raise KeyError")

try:
    load(sample, projections=[("a", "b")], layout={("a", "b"): (0, 0)}, group="missing")
except KeyError as err:
    assert "Group column" in str(err)
else:
    raise AssertionError("Expected missing group column to raise KeyError")

try:
    rp.plot(rug_edges="diagonal")
except ValueError as err:
    assert "rug_edges" in str(err)
else:
    raise AssertionError("Expected invalid rug_edges to raise ValueError")

try:
    rp.with_highlight("not-a-row").plot()
except KeyError as err:
    assert "highlight" in str(err)
else:
    raise AssertionError("Expected invalid highlight to raise KeyError")